# Q - Do groups re-post or repurpose older leaks?

Are the Jan - Sep 2026 victims of our five groups (akira, incransom, play, qilin, safepay) really new breaches, or old leaks and public breaches posted again? Answered **only from data this project scraped**:

| source | what it holds |
|---|---|
| `data/victims.json` | 2026 posts with full details (claim, size, file count, proof ids) |
| `data/history.json` | 2023 - 2025 posts with the same details, from `rw backfill --since 2023-01-01 --pages 400` |
| `posts.json` | legacy post **titles** only - fills the periods the leak sites no longer host |

Two steps:

1. **Details** - is the same company posted again, and does the claimed data match?
2. **Hashes** - do the leak artefacts hash to the same value across posts?

**Limits:**

- The leak sites don't keep everything. Play only goes back to Jul 2025, qilin to Mar 2025, incransom to Apr 2024 and safepay to Aug 2025. Older posts from those groups are titles only (from `posts.json`), and safepay has no older posts at all.
- We hold **no stolen files** and don't download them. The hashes are of what groups publish about a leak: claim text, descriptions, size and file count, **proof file ids** (qilin names its proof photos with a 32-hex digest, incransom gives each proof a CDN id), and Play download paths and archive passwords. The proof images themselves are never fetched.

In [ ]:
import os
import re
import glob
import json
import hashlib
import difflib
from collections import defaultdict

import pandas as pd
import tldextract
from bs4 import BeautifulSoup

if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

TRACKED = {g['name'] for g in json.load(open('groups.json', encoding='utf-8'))}

def load(path, source):
    df = pd.DataFrame(json.load(open(path, encoding='utf-8')))
    df['date'] = pd.to_datetime(df['date'])
    df['source'] = source
    return df

victims = load('data/victims.json', 'victims.json')
history = load('data/history.json', 'history.json') if os.path.exists('data/history.json') else victims.iloc[0:0]

posts = pd.DataFrame(json.load(open('posts.json', encoding='utf-8')))
posts = posts.rename(columns={'group_name': 'group', 'post_title': 'victim'})
posts['date'] = pd.to_datetime(posts['discovered'].str[:10])
posts['source'] = 'posts.json'

# titles only count where the leak site no longer has the post - i.e. before the oldest backfilled post
oldest = history.groupby('group').date.min()
titles = posts[posts.group.isin(TRACKED) & (posts.date >= '2023-01-01') & (posts.date < '2026-01-01')]
titles = titles[titles.date < titles.group.map(oldest).fillna(pd.Timestamp('2026-01-01'))]
earlier = pd.concat([history, titles], ignore_index=True)

print(len(victims), '2026 posts |', len(history), 'detailed 2023-25 posts |', len(titles), 'title-only 2023-25 posts')
pd.crosstab([earlier.group], [earlier.date.dt.year, earlier.source]).reindex(sorted(TRACKED), fill_value=0)

### Before comparing: dates that are not attack dates

Two things in the backfill change how dates should be read:

- **Akira** announces a victim in its news feed and leaks the data in a separate feed weeks later. With only 2026 history, a victim announced in late 2025 and leaked in 2026 was dated 2026. With the deep backfill, 74 of those rows now carry their 2025 announcement date and move to `history.json`. That is the same group following through on its own leak, not a re-post.
- **Safepay** dates hundreds of posts within a few days in Aug 2025, all with very high view counts. That looks like the site being re-uploaded or migrated: the dates are upload dates, not attack dates.

In [ ]:
per_day = history.groupby(['group', history.date.dt.date]).size().rename('posts').reset_index()
per_day.sort_values('posts', ascending=False).head(8)

## 1. Same company posted again

Same normalised key as `rebrands_check.ipynb`: the domain label if there is a domain, otherwise the name without boilerplate and company suffixes. Each post is keyed by both its website and its name.

In [ ]:
PREFIXES = re.compile(r'^(leaks? (from|of) (the )?(company )?|leakage from (the )?company |data of )', re.I)
SUFFIXES = {'inc', 'llc', 'ltd', 'limited', 'gmbh', 'ag', 'sa', 'sas', 'srl', 'spa', 'bv', 'nv', 'plc', 'corp',
            'corporation', 'co', 'company', 'group', 'holdings', 'holding', 'pty', 'sdn', 'bhd', 'kk', 'ltda', 'the'}
DOMAIN = re.compile(r'([a-z0-9-]+\.)+[a-z]{2,}')
# website keys that are lookup sites, not the victim
NOT_VICTIM = {'zoominfo', 'linkedin', 'google', 'facebook'}

def victim_key(name, website=None):
    text = str(website if isinstance(website, str) and website else name or '').lower().strip()
    match = DOMAIN.search(text)
    if match:
        ext = tldextract.extract(match.group(0))
        if ext.domain and ext.suffix and len(ext.domain) >= 4:
            return None if ext.domain in NOT_VICTIM else ext.domain
    text = PREFIXES.sub('', text)
    words = [w for w in re.split(r'[^a-z0-9]+', text) if w and w not in SUFFIXES]
    key = ''.join(words)
    return key if len(key) >= 5 else None

def keys(df):
    website = df.website if 'website' in df else pd.Series(None, index=df.index)
    return [{victim_key(n, w), victim_key(n)} - {None} for n, w in zip(df.victim, website)]

victims['keys'] = keys(victims)
earlier['keys'] = keys(earlier)
posts['keys'] = keys(posts)

def earlier_posts(pool, label):
    flat = pool.explode('keys').dropna(subset=['keys'])
    rows = []
    for i, v in victims.iterrows():
        for j, o in flat[flat['keys'].isin(v['keys']) & (flat.date < v.date)].iterrows():
            rows.append({'scope': label, 'idx_2026': i, 'idx_earlier': j, 'group_2026': v.group, 'victim_2026': v.victim,
                         'date_2026': v.date.date(), 'earlier_group': o.group, 'earlier_name': o.victim,
                         'earlier_date': o.date.date(), 'earlier_source': o.source,
                         'gap_days': (v.date - o.date).days, 'same_group': v.group == o.group})
    return pd.DataFrame(rows).drop_duplicates(['idx_2026', 'earlier_group'])

reposts = earlier_posts(earlier, 'five groups 2023-25')
print(len(reposts), 'matches |', reposts.idx_2026.nunique(), 'of', len(victims), '2026 victims were posted by one of the five in 2023-25')
reposts.drop(columns=['idx_2026', 'idx_earlier'])

In [ ]:
# the same company twice inside 2026
in_2026 = victims.explode('keys').dropna(subset=['keys']).drop_duplicates(['keys', 'group', 'victim'])
dupes = in_2026[in_2026.groupby('keys').victim.transform('size') > 1]
dupes[['keys', 'group', 'victim', 'website', 'date', 'data_size', 'data_files']]

**Extra, out of scope:** the same lookup against every *other* group in `posts.json`, any year. This is still our own scraped data. **Check each row by eye.** A name match can be a different company or a subsidiary.

In [ ]:
other = posts[~posts.group.isin(TRACKED)]
elsewhere = earlier_posts(other, 'other groups, any year').sort_values('gap_days')
elsewhere.drop(columns=['idx_2026', 'idx_earlier'])

### Does the claimed data match?

For matches where the earlier post also has details, compare size, file count, claim text and proof ids. A recycled leak would show the **same** size and file count, or overlapping proof ids. A new breach shows different numbers and different proofs. Claims flagged `leak_claim_generic` are a group's template text and don't count as a match.

In [ ]:
def claim_text(r):
    if r.get('leak_claim_generic') is True:
        return None
    return r.get('leak_claim') if isinstance(r.get('leak_claim'), str) else None

def ids(value):
    return set(value.split(';')) if isinstance(value, str) else set()

rows = []
for _, m in reposts.iterrows():
    a, b = victims.loc[m.idx_2026], earlier.loc[m.idx_earlier]
    ca, cb = claim_text(a), claim_text(b)
    rows.append({**m.drop(['idx_2026', 'idx_earlier']).to_dict(),
                 'size_2026': a.get('data_size'), 'size_earlier': b.get('data_size'),
                 'files_2026': a.get('data_files'), 'files_earlier': b.get('data_files'),
                 'same_size': bool(a.get('data_size')) and a.get('data_size') == b.get('data_size'),
                 'claim_similarity': round(difflib.SequenceMatcher(None, ca, cb).ratio(), 2) if ca and cb else None,
                 'shared_proofs': len(ids(a.get('proof_ids')) & ids(b.get('proof_ids')))})
compare = pd.DataFrame(rows)
compare

## 2. Hash comparison of leak artefacts

Every post from 2023 - 2026 that has details is pooled. Each artefact is normalised (lower case, punctuation dropped) and hashed with SHA-256. One hash under **different** posts means the same text, numbers or proof file appear twice. Group templates are expected to collide, so read the `example` column.

In [ ]:
pool = pd.concat([history, victims], ignore_index=True)
pool['post'] = pool.group + '|' + pool.victim

def norm(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()

def sha(text):
    return hashlib.sha256(norm(text).encode()).hexdigest()[:16]

def shared_hashes(df, column, value=None):
    d = df.dropna(subset=[column]).copy()
    d['hash'] = d[column].map(sha) if value is None else d.apply(value, axis=1)
    out = d.groupby('hash').agg(posts=('post', 'nunique'), groups=('group', lambda s: ', '.join(sorted(set(s)))),
                                first=('date', 'min'), last=('date', 'max'), example=(column, 'first'))
    print('%-12s %5d posts | %5d distinct hashes | %d shared by more than one post'
          % (column, len(d), d.hash.nunique(), (out.posts > 1).sum()))
    return out[out.posts > 1].sort_values('posts', ascending=False)

claims = shared_hashes(pool, 'leak_claim')
descriptions = shared_hashes(pool, 'description')
sizes = shared_hashes(pool[pool.data_files.fillna(0) > 0], 'data_files',
                      lambda r: sha('%s|%d' % (r.data_size, r.data_files)))
hashes = pd.concat({'leak_claim': claims, 'description': descriptions, 'size+files': sizes})
hashes.groupby(level=0).head(6)

In [ ]:
# proof files - the same proof id under two posts is the same file shown again
proofs = pool.dropna(subset=['proof_ids']).assign(proof=lambda d: d.proof_ids.str.split(';')).explode('proof')
reused = proofs[proofs.groupby('proof').post.transform('nunique') > 1]
print(proofs.proof.nunique(), 'proof files across', proofs.post.nunique(), 'posts |',
      reused.proof.nunique(), 'appear under more than one post')
reused.groupby('proof').agg(posts=('post', lambda s: ' / '.join(sorted(set(s)))),
                            dates=('date', lambda s: ', '.join(sorted({str(d.date()) for d in s}))))

In [ ]:
# play post pages carry download links & a rar password - the closest thing we have to the archive itself
links, passwords = defaultdict(set), defaultdict(set)
for path in glob.glob('source/detail/play-*.html'):
    text = BeautifulSoup(open(path, encoding='utf-8', errors='ignore').read(), 'html.parser').get_text(' ', strip=True)
    name = text.split(' PLAY ')[0]
    for archive in re.findall(r'[a-z2-7]{56}\.onion/(\S+)', text):
        links[sha(archive)].add(name)
    secret = re.search(r'password:\s*(\S+)', text, re.I)
    if secret:
        passwords[sha(secret.group(1))].add(name)
print(len(links), 'archive paths |', sum(len(v) > 1 for v in links.values()), 'reused across victims')
print(len(passwords), 'archive passwords |', sum(len(v) > 1 for v in passwords.values()), 'reused across victims')

In [ ]:
os.makedirs('analysis/output', exist_ok=True)
compare.to_csv('analysis/output/repurposed_matches.csv', index=False)
elsewhere.drop(columns=['idx_2026', 'idx_earlier']).to_csv('analysis/output/repurposed_matches_other_groups.csv', index=False)
hashes.to_csv('analysis/output/repurposed_shared_hashes.csv')
reused.to_csv('analysis/output/repurposed_shared_proofs.csv', index=False)
print('written to analysis/output/')

## Reading the results

- **Re-posting is uncommon.** 11 of 1,706 victims from 2026 (~0.6%) had been posted by one of the five groups in 2023 - 25, and no company is posted twice within 2026. Gericke is two different firms (gerickegroup.com vs gericke-spedition.de), which leaves 10.
- **Same group, same company again (3):** Play - Smoker's Choice (2024-10 -> 2026-03), Qilin - RMZ Oilfield (2025-06 -> 2026-03) and Qilin - Cary Pediatric Center (2025-09 -> 2026-01). **Cary Pediatric is the strongest recycling candidate**: both posts claim 130 GB with 12 proof photos, but the photo ids differ. Either the proofs were re-uploaded (new names) or they are different files. We can't tell which without downloading them.
- **Different group, same company (7):** EFU Life and Jubilee Jobs (incransom -> qilin), ALLMAX (incransom -> play), GDZ (play "full" 118 GB leak in 2025-09 -> qilin 2026-01, and lockbit3 in 2023), Air Int'l Thermal and Guntert & Zimmerman (play -> qilin), Compact Mould (play -> safepay). Gaps run from 4 months to 2.7 years. Where both sides have a size, the sizes differ, and no proofs are shared. That fits a second breach or an affiliate moving on better than the same dump re-posted.
- **Other gangs' old victims (extra):** about 20 victims from 2026 had been posted by groups outside the five, mostly 1.5 - 4.5 years earlier (ransomhub, lockbit, hunters, 8base, blackbasta...). Ransomhub victims reappear 15 - 16 months later, after ransomhub went quiet. Check these rows by eye.
- **Hashes:** no proof file id (16k files across 1,233 posts), Play archive path, archive password or size + file count appears under two posts. Every shared claim or description hash is group boilerplate (Play's two template claims cover 237 posts), or one firm posted several times on the same day.
- **Dates that are not attack dates:** 74 Akira victims stored as 2026 were announced in 2025 and leaked in 2026 (median gap 94 days). That is a delayed leak, not a re-post, and they now sit in `history.json`. Safepay dates ~270 posts to 2 - 9 Aug 2025, which is a site re-upload, so their dates say nothing about when the attack happened.
- **What would settle it:** comparing the proof images (or the leaked files) by content. That means downloading victims' stolen data, which we deliberately don't do. Short of that, the same size + file count + proof count across two posts (as with Cary Pediatric) is the best signal the scraped data gives.